In [0]:
from pyspark.sql.functions import col, map_filter, size, create_map, lit
from pyspark.sql.types import (
    StructType, StructField, StringType,
    ArrayType, MapType, IntegerType, LongType
)

RAW_PATH    = "s3://s3-goodreads-data/Raw"
BRONZE_PATH = "s3://s3-goodreads-data/Bronze/books"

books_schema = StructType([
    StructField("book_id",            StringType(), True),
    StructField("title",              StringType(), True),
    StructField("authors", ArrayType(StructType([
        StructField("author_id", StringType(), True),
        StructField("role",      StringType(), True)
    ])), True),
    StructField("average_rating",     StringType(), True),
    StructField("format",             StringType(), True),
    StructField("image_url",          StringType(), True),
    StructField("is_ebook",           StringType(), True),
    StructField("language_code",      StringType(), True),
    StructField("num_pages",          StringType(), True),
    StructField("publication_year",   StringType(), True),
    StructField("publisher",          StringType(), True),
    StructField("ratings_count",      StringType(), True),
    StructField("series",             ArrayType(StringType()), True),
    StructField("text_reviews_count", StringType(), True),
])

genres_schema = StructType([
    StructField("book_id",                                StringType(), True),
    StructField("children",                               LongType(),   True),
    StructField("comics, graphic",                        LongType(),   True),
    StructField("fantasy, paranormal",                    LongType(),   True),
    StructField("fiction",                                LongType(),   True),
    StructField("history, historical fiction, biography", LongType(),   True),
    StructField("mystery, thriller, crime",               LongType(),   True),
    StructField("non-fiction",                            LongType(),   True),
    StructField("poetry",                                 LongType(),   True),
    StructField("romance",                                LongType(),   True),
    StructField("young-adult",                            LongType(),   True),
])

columns_to_drop = [
    "asin", "kindle_asin", "isbn", "isbn13", "country_code",
    "link", "url", "publication_day", "publication_month",
    "edition_information", "work_id", "title_without_series",
    "description", "popular_shelves", "similar_books"
]

genre_cols = [
    "children", "comics, graphic", "fantasy, paranormal", "fiction",
    "history, historical fiction, biography", "mystery, thriller, crime",
    "non-fiction", "poetry", "romance", "young-adult"
]

# Build map directly from struct fields
genre_map_expr = create_map(*[
    x for genre in genre_cols
    for x in (lit(genre), col(f"`{genre}`").cast(IntegerType()))
])

df_books_raw  = spark.read.schema(books_schema).json(f"{RAW_PATH}/metadata/books.json.gz")
df_genres_raw = spark.read.schema(genres_schema).json(f"{RAW_PATH}/metadata/genres.json.gz")

df_final = df_books_raw.drop(*columns_to_drop) \
    .join(df_genres_raw, on="book_id", how="left") \
    .withColumn("genres", map_filter(genre_map_expr, lambda k, v: v > 0)) \
    .drop(*genre_cols) \
    .filter(size(col("genres")) > 0)

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(BRONZE_PATH)

print("✅ Books Bronze done.")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from concurrent.futures import ThreadPoolExecutor, as_completed

RAW_PATH    = "s3://s3-goodreads-data/Raw"
BRONZE_PATH = "s3://s3-goodreads-data/Bronze"
RAW_PARQUET = f"{RAW_PATH}/interactions/parquet"

interactions_schema = StructType([
    StructField("book_id",    StringType(), True),
    StructField("date_added", StringType(), True),
    StructField("review_id",  StringType(), True),
    StructField("user_id",    StringType(), True),
])

def process_interactions():
    spark.read.schema(interactions_schema).parquet(RAW_PARQUET) \
        .dropDuplicates(["user_id", "book_id"]) \
        .write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{BRONZE_PATH}/interactions")
    return "✅ Bronze Interactions done."

def process_authors():
    spark.read.json(f"{RAW_PATH}/metadata/authors.json.gz") \
        .write.format("delta") \
        .mode("overwrite") \
        .save(f"{BRONZE_PATH}/authors")
    return "✅ Bronze Authors done."

# Run both in parallel
with ThreadPoolExecutor(max_workers=2) as ex:
    futures = [ex.submit(f) for f in [process_interactions, process_authors]]
    for f in as_completed(futures):
        print(f.result())

In [0]:
from pyspark.sql.functions import col, count, when

BRONZE_PATH = "s3://s3-goodreads-data/Bronze"

def format_number(n):
    """Formats numbers into K (Thousands) or M (Millions)."""
    if n >= 1_000_000:
        return f"{n/1_000_000:.1f}M"
    elif n >= 1_000:
        return f"{n/1_000:.1f}K"
    else:
        return str(n)

# Define tables and their formats
tables = [
    {"name": "interactions", "format": "delta"},
    {"name": "books",        "format": "delta"},
    {"name": "authors",      "format": "delta"}
]


# 2. RUN HEALTH CHECK
print(f"{'TABLE':<15} | {'TOTAL ROWS':<10} | {'NULLS (Per Column)'}")
print("-" * 80)

for t in tables:
    path = f"{BRONZE_PATH}/{t['name']}"
    
    try:
        # Read the table
        df = spark.read.format(t['format']).load(path)
        
        # 1. Count Total Rows
        total_rows = df.count()
        formatted_rows = format_number(total_rows)
        
        # 2. Count Nulls
        null_exprs = [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
        null_counts = df.select(*null_exprs).first().asDict()
        
        # 3. Format Null Output
        null_report = []
        for col_name, null_val in null_counts.items():
            if null_val > 0:
                null_report.append(f"{col_name}: {format_number(null_val)}")
        
        null_text = ", ".join(null_report) if null_report else "✅ Perfect (0 Nulls)"
        
        # Print Row
        print(f"{t['name']:<15} | {formatted_rows:<10} | {null_text}")
        
    except Exception as e:

            print(f"{t['name']:<15} | ERROR | Could not read table ({e})")

In [0]:
from pyspark.sql.functions import col, count, desc

# 1. CONFIGURATION
BRONZE_PATH = "s3://s3-goodreads-data/Bronze"

df_books = spark.read.format("delta").load(f"{BRONZE_PATH}/books") 

# 2. CHECK FOR DUPLICATES

# Group by ID and count how many times each appears
duplicate_check = df_books.groupBy("book_id") \
                          .count() \
                          .filter(col("count") > 1) \
                          .orderBy(desc("count"))

dup_count = duplicate_check.count()

# 3. REPORT & PREVIEW
print("-" * 40)
print(f"Total Books:     {df_books.count():,}")
print(f"Duplicate IDs:   {dup_count:,}")
print("-" * 40)

if dup_count > 0:
    display(duplicate_check.limit(10))

    bad_id = duplicate_check.first()['book_id']
    display(df_books.filter(col("book_id") == bad_id))
else:
    print("No duplicates found.")

In [0]:
from pyspark.sql.functions import col, count

BRONZE_PATH = "s3://s3-goodreads-data/Bronze"
INTERACTIONS_PATH = f"{BRONZE_PATH}/interactions"

df_interactions = spark.read.format("delta").load(INTERACTIONS_PATH)

# 2. FIND DUPLICATES
# grouping by User + Book to find how many times they appear.

df_duplicates = df_interactions.groupBy("user_id", "book_id") \
    .agg(count("*").alias("duplicate_count")) \
    .filter(col("duplicate_count") > 1) \
    .orderBy(col("duplicate_count").desc())

# 3. DISPLAY
count_dupes = df_duplicates.count()
print(f"⚠️ Found {count_dupes} pairs of (User + Book) that appear more than once.")

if count_dupes > 0:
    print("Here are the top offenders:")
    display(df_duplicates.limit(20))